# 1. Generación de Datos Sintéticos de Tráfico

Generar datos sintéticos realistas simulando patrones diarios de tráfico en Guadalajara.

## Objetivo
- Crear dataset de prueba con comportamiento realista
- Simular patrones diarios y congestiones
- Validar estructura del pipeline antes de datos reales

In [4]:
# ============================================================================
# IMPORTAR LIBRERÍAS
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import os

# Configuración de visualización
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
sns.set_palette('husl')

print('✓ Librerías cargadas')

✓ Librerías cargadas


## Configuración de Parámetros

Definir rangos realistas para variables de tráfico:

In [5]:
# ============================================================================
# CONFIGURACIÓN
# ============================================================================
np.random.seed(42)

# Parámetros de generación
PARAMS = {
    'velocity_min': 20,      # km/h
    'velocity_max': 80,      # km/h
    'density_min': 5,        # veh/km
    'density_max': 100,      # veh/km
    'flow_min': 300,         # veh/h
    'flow_max': 2000,        # veh/h
    'wait_time_max': 300,    # segundos
    'num_roads': 5,
    'num_days': 7,
    'obs_per_day': 144       # Cada 10 minutos
}

# Avenidas principales de Guadalajara
ROADS = [
    'Av. Chapultepec',
    'Av. México',
    'Av. Vallarta',
    'Av. Gigantes',
    'Av. López Mateos'
]

print(f'✓ Configuración: {PARAMS["num_roads"]} avenidas, {PARAMS["num_days"]} días')

✓ Configuración: 5 avenidas, 7 días


## Funciones de Generación

Crear patrones diarios y datos correlacionados:

In [6]:
# ============================================================================
# FUNCIONES
# ============================================================================
def get_daily_pattern(hour):
    """
    Factor multiplicador según hora del día.
    Simula horas pico (7-9 AM, 5-7 PM).
    """
    if 7 <= hour < 9:
        return 1.3  # Pico matutino
    elif 17 <= hour < 19:
        return 1.4  # Pico vespertino
    elif 12 <= hour < 13:
        return 1.1  # Mediodía
    else:
        return 1.0  # Normal

def generate_traffic_data(params):
    """
    Generar dataset sintético de tráfico.
    Retorna DataFrame con variables correlacionadas.
    """
    records = []
    start_date = datetime(2024, 1, 1)
    
    for day in range(params['num_days']):
        current_date = start_date + timedelta(days=day)
        
        for road_idx, road in enumerate(ROADS[:params['num_roads']]):
            for obs in range(params['obs_per_day']):
                # Calcular hora del día
                minutes = obs * 10
                hour = minutes // 60
                
                # Factor de patrón diario
                daily_factor = get_daily_pattern(hour)
                
                # Generar velocidad con patrón
                velocity = np.random.normal(50, 10) * daily_factor
                velocity = np.clip(velocity, params['velocity_min'], params['velocity_max'])
                
                # Generar densidad (inversa a velocidad)
                density = np.random.normal(40, 15) / daily_factor
                density = np.clip(density, params['density_min'], params['density_max'])
                
                # Flujo = f(velocidad, densidad)
                flow = (velocity * density / 1000) * 100
                flow = np.clip(flow, params['flow_min'], params['flow_max'])
                
                # Tiempo de espera inverso a velocidad
                wait_time = (1 - velocity / params['velocity_max']) * params['wait_time_max']
                wait_time = max(0, wait_time + np.random.normal(0, 10))
                
                # Contar detenciones
                stops = np.random.poisson(wait_time / 30)
                
                timestamp = current_date + timedelta(minutes=minutes)
                
                records.append({
                    'timestamp': timestamp,
                    'road': road,
                    'hour': hour,
                    'velocity_kmh': round(velocity, 2),
                    'density_veh_km': round(density, 2),
                    'flow_veh_h': round(flow, 2),
                    'wait_time_sec': round(wait_time, 2),
                    'stops_count': stops
                })
    
    return pd.DataFrame(records)

## Generación de Datos

In [7]:
# ============================================================================
# GENERAR DATOS
# ============================================================================
print(f"Generando {PARAMS['num_days']} días de datos...")
df = generate_traffic_data(PARAMS)
print(f"✓ {len(df):,} registros generados")
print(f"\nPrimeros registros:")
print(df.head())
print(f"\nÚltimos registros:")
print(df.tail())

Generando 7 días de datos...
✓ 5,040 registros generados

Primeros registros:
            timestamp             road  hour  velocity_kmh  density_veh_km  \
0 2024-01-01 00:00:00  Av. Chapultepec     0         54.97           37.93   
1 2024-01-01 00:10:00  Av. Chapultepec     0         65.23           63.69   
2 2024-01-01 00:20:00  Av. Chapultepec     0         44.19           32.12   
3 2024-01-01 00:30:00  Av. Chapultepec     0         40.76           26.38   
4 2024-01-01 00:40:00  Av. Chapultepec     0         38.49           45.64   

   flow_veh_h  wait_time_sec  stops_count  
0      300.00         100.35            1  
1      415.44          63.06            0  
2      300.00         128.57            4  
3      300.00         133.03            5  
4      300.00         149.66            4  

Últimos registros:
               timestamp              road  hour  velocity_kmh  \
5035 2024-01-07 23:10:00  Av. López Mateos    23         42.24   
5036 2024-01-07 23:20:00  Av. López M

## Validación Estadística

In [8]:
# ============================================================================
# VALIDACIÓN
# ============================================================================
print("\nEstadísticas descriptivas:")
stats = df[['velocity_kmh', 'density_veh_km', 'flow_veh_h', 'wait_time_sec']].describe()
print(stats)

print("\nCorrelaciones:")
corr = df[['velocity_kmh', 'density_veh_km', 'flow_veh_h', 'wait_time_sec']].corr()
print(corr)


Estadísticas descriptivas:
       velocity_kmh  density_veh_km   flow_veh_h  wait_time_sec
count   5040.000000     5040.000000  5040.000000    5040.000000
mean      52.704127       38.570540   306.943456     102.536347
std       11.861973       14.950616    24.336894      45.297484
min       20.000000        5.000000   300.000000       0.000000
25%       44.607500       27.950000   300.000000      74.270000
50%       51.775000       38.170000   300.000000     105.515000
75%       60.000000       48.325000   300.000000     133.170000
max       80.000000       99.130000   610.900000     239.950000

Correlaciones:
                velocity_kmh  density_veh_km  flow_veh_h  wait_time_sec
velocity_kmh        1.000000       -0.130882    0.250548      -0.975967
density_veh_km     -0.130882        1.000000    0.466757       0.128639
flow_veh_h          0.250548        0.466757    1.000000      -0.242329
wait_time_sec      -0.975967        0.128639   -0.242329       1.000000


## Exportación

In [9]:
# ============================================================================
# EXPORTAR
# ============================================================================
output_path = Path('../data/raw')
output_path.mkdir(parents=True, exist_ok=True)

csv_file = output_path / 'synthetic_traffic.csv'
df.to_csv(csv_file, index=False)
print(f"✓ Datos exportados: {csv_file}")
print(f"  Tamaño: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

✓ Datos exportados: ..\data\raw\synthetic_traffic.csv
  Tamaño: 612.41 KB
